In [1]:
from sentence_transformers import SentenceTransformer

/opt/homebrew/anaconda3/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import faiss

In [3]:
from pathlib import Path
from llama_index.readers.file import PyMuPDFReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode

In [4]:
from llama_index.core import SimpleDirectoryReader

In [5]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

In [6]:
embed_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

In [7]:
import os
from langchain.document_loaders import PyMuPDFLoader

pdf_folder = 'pdfs'
documents = []

for filename in os.listdir(pdf_folder):
    if filename.endswith('.pdf'):
        file_path = os.path.join(pdf_folder, filename)
        # Use PyMuPDFLoader to read the PDF
        loader = PyMuPDFLoader(file_path)
        docs = loader.load()
        documents.extend(docs)

print(f"Loaded {len(documents)} documents from PDFs.")

Loaded 687 documents from PDFs.


In [8]:
# splitting documents
text_parser = SentenceSplitter(chunk_size=1024)

In [9]:
len(documents) #this are the sum of all pages from all pdfs

687

In [10]:
text_chunks = []
# maintain relationship with source doc index, to help inject doc metadata in (3)
doc_idxs = []
for doc_idx, doc in enumerate(documents):
    cur_text_chunks = text_parser.split_text(doc.page_content)
    text_chunks.extend(cur_text_chunks)
    doc_idxs.extend([doc_idx] * len(cur_text_chunks))

In [11]:
#setting nodes for chunks of text
nodes = []
for idx, text_chunk in enumerate(text_chunks):
    node = TextNode(text=text_chunk)
    src_doc = documents[doc_idxs[idx]]
    node.metadata = src_doc.metadata
    nodes.append(node)

In [12]:
embed_model.encode(nodes[1].text).shape

(1024,)

In [13]:

# for node in nodes:
#     content = node.get_content(metadata_mode="all")
#     if not hasattr(node, 'embedding') or node.embedding is None:
#         node_embedding = embed_model.encode(content)
#         node.embedding = list(node_embedding)

# # For text chunks, assuming you maintain a list or dict for embeddings
# if 'chunks_embeddings' not in locals():
#     chunks_embeddings = embed_model.encode(text_chunks)

In [14]:
#embedding each node 
for node in nodes:
    node_embedding = embed_model.encode(node.get_content(metadata_mode="all"))
    node.embedding = list(node_embedding)

In [15]:
chuncks_embeddings = embed_model.encode(text_chunks)

In [16]:
chuncks_embeddings.shape

(1096, 1024)

In [17]:
# Step 2: Indexing with FAISS
index = faiss.IndexFlatL2(chuncks_embeddings.shape[1])
index.add(chuncks_embeddings)

In [18]:
# Step 3: Retrieval

def search(query: str, k: int = 3 ):
    """a function that embeds a new query and returns the most probable results"""
    query_embedding = embed_model.encode([query]) # embed new query
    distances, indices = index.search(query_embedding, k)
    nearest_documents = [text_chunks[i] for i in indices[0]]
    return distances, nearest_documents 


query = "How did the economy behave during the first quarter of 2020?"
distances, nearest_documents  = search(query)

### Llama 3 model

In [19]:
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# import torch
# torch.set_default_tensor_type('torch.FloatTensor')  # For CPU
# if torch.cuda.is_available():
#     torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [20]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [21]:
# # use quantization to lower GPU usage
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
# )

# tokenizer = AutoTokenizer.from_pretrained(model_id)
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = AutoModelForCausalLM.from_pretrained(model_id, device=device)
# # model = AutoModelForCausalLM.from_pretrained(
# #     model_id,
# #     torch_dtype=torch.bfloat16,
# #     device_map="auto",
# #     quantization_config=bnb_config
# # )
# terminators = [
#     tokenizer.eos_token_id,
#     tokenizer.convert_tokens_to_ids("<|eot_id|>")
# ]

In [26]:
SYS_PROMPT = """You are an assistant for answering questions.
You are given the extracted parts of a long document and a question. Provide a conversational answer.
If you don't know the answer, just say "I do not know." Don't make up an answer. """

def format_prompt(prompt, retrieved_documents, k):
  """using the retrieved documents we will prompt the model to generate our responses"""
  PROMPT = f"Question:{prompt}\nContext:"
  for idx in range(k) :
    PROMPT+= f"{retrieved_documents[idx]}\n"
  return PROMPT

def generate(formatted_prompt):
  formatted_prompt = formatted_prompt[:2000] # to avoid GPU OOM
  messages = [{"role":"system","content":SYS_PROMPT},{"role":"user","content":formatted_prompt}]
  # tell the model to generate
  terminators = tokenizer.eos_token_id

  input_ids = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      return_tensors="pt"
  ).to(model.device)
  outputs = model.generate(
      input_ids,
      max_new_tokens=1024,
      eos_token_id=terminators,
      do_sample=True,
      temperature=0.6,
      top_p=0.9,
  )
  response = outputs[0][input_ids.shape[-1]:]
  return tokenizer.decode(response, skip_special_tokens=True)

def rag_chatbot(prompt:str,k:int=2):
  distances , retrieved_documents = search(prompt, k)
  formatted_prompt = format_prompt(prompt, retrieved_documents,k)
  return generate(formatted_prompt)


In [27]:
distances, retrieved_documents = search(query, 2)

In [28]:
query

'How did the economy behave during the first quarter of 2020?'

In [29]:
rag_chatbot(query, k = 2)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


KeyboardInterrupt: 

In [ ]:
query = "¿Cómo puedo conocer el ESTADO DE TRAMITACIÓN de mi expediente de nacionalidad?"
rag_chatbot(query, k = 5)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'Para conocer el estado de tramitación de tu expediente de nacionalidad, puedes hacerlo de la siguiente manera:\n\n Primero, debes verificar si tienes una cuenta en el registro electrónico único del Ministerio de Justicia. Si no tienes cuenta, puedes crear una en el sitio web del Ministerio de Justicia.\n\nUna vez tengas acceso a tu cuenta, puedes ingresar a la sección de "Mi expediente" y buscar tu expediente de nacionalidad. Allí, podrás ver el estado de tramitación actualizado.\n\nTambién puedes comunicarte con el Ministerio de Justicia a través del correo postal o del Registro Electrónico del Ministerio de Justicia (apartado de escritos y solicitudes de la sede electrónica), ubicado en la dirección Calle La Bolsa 8, 28012 Madrid. Debes incluir tu número de expediente R y proporcionar la información necesaria para que puedan verificar el estado de tramitación de tu expediente.\n\nRecuerda que es importante tener en cuenta que, según la normativa vigente, no se puede realizar una cop